# 实验 3：两层网络的参数化、特征变化与宽度迁移

**对应第5讲；课堂实验约1.5学时。** 本 Notebook 调用 `experiments/lab03_width.py`，比较同一套原始参数坐标下的 NTP 和 mean-field 配对尺度。默认一个随机种子先运行，后面提供默认关闭的三种子扩展。

核心问题是：相同的初始函数与有效核，是否意味着相同的特征学习轨迹？本实验同时记录验证损失、原始参数位移、隐藏特征Gram漂移和有效核变化。特征漂移大并不自动表示特征更好，单种子的有限宽度扫描也不建立渐近定理。


## 环境与执行方式

在已有的 Jupyter 或 VS Code Notebook 环境中，选择安装了课程依赖的 Python 内核，按顺序运行单元格。计算依赖与命令行脚本相同；展示使用 Notebook 环境已有的 `IPython.display`，不需要 pandas 或额外的交互组件。

如尚未安装依赖，请在**课程目录的终端**手动运行：

```bash
python -m pip install -r requirements.txt
```

本 Notebook 不会自动安装包，也不会下载外部数据集。生成的图、CSV 和 JSON 写入课程目录下的 `results/notebooks/`，不会写入 Notebook 的源码单元格。仓库版本有意保留空输出；运行后可以通过生成文件复核结果。


## 课前：前向式、存储变量、学习率必须配对

脚本存储原始 $a_r,w_r$，使用无动量SGD与平均半平方损失：

| 约定 | 原始前向式 | 原始参数学习率 |
|---|---|---|
| NTP | $f=m^{-1/2}\sum_r a_r\tanh(w_r^\top x)$ | $\eta$ |
| mean-field（MF） | $f=m^{-1}\sum_r a_r\tanh(w_r^\top x)$ | $m\eta$ |

两个参数组初始化的量级都不随宽度改变。实际训练输出减去由**冻结初始参数**计算的初始函数，因此所有初始预测为零，而对可训练参数的Jacobian不变。

运行前回答：

1. 为什么两种约定具有相同的初始有效核，却可有不同的每神经元移动？
2. 如果把MF实际输出权重直接存成 $b=a/m$，为什么对应SGD步长是 $\eta/m$ 而不是 $m\eta$？
3. 为什么比较各宽度特征漂移时应固定同一个基础学习率，而不是为每个宽度另外选最优步长？
4. 迁移成功的容忍量是什么？应在查看验证或测试结果之前规定怎样的判据？


In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import json
import sys
from IPython.display import display, Image, JSON, Markdown

# 可从 notebooks/、课程目录或仓库根目录启动，不依赖机器上的绝对路径。
start = Path.cwd().resolve()
candidates = []
for parent in (start, *start.parents):
    candidates.extend((parent, parent / '2027' / 'nonlinear optimization'))
COURSE_ROOT = next(
    (p for p in candidates
     if (p / 'experiments' / 'lab01_sgd.py').is_file()
     and (p / 'chapters').is_dir()),
    None,
)
if COURSE_ROOT is None:
    raise FileNotFoundError('未找到课程目录：请从本仓库的课程目录或 notebooks 目录启动。')
EXPERIMENTS = COURSE_ROOT / 'experiments'
if str(EXPERIMENTS) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS))
print('课程目录：', COURSE_ROOT)
print('实验实现：', EXPERIMENTS)


In [ ]:
from lab03_width import Config as LabConfig, run as run_lab

# 保留预先声明的全部6个候选，不在看到失败后删除大步长。
config = LabConfig(
    seed=2027, steps=60, batch=32, widths=(32, 64, 128),
    rates=(0.003, 0.01, 0.03, 0.1, 0.3, 1.0),
    device='auto', diagnostic_epsilon=1e-12,
)
# 预先声明：在当前有限网格内，比该宽度自身扫描最优值最多高0.05。
TRANSFER_ABS_TOL = 0.05
OUTPUT = COURSE_ROOT / 'results' / 'notebooks' / 'lab03' / f'seed_{config.seed}'
display(JSON({
    'config': asdict(config),
    'transfer_absolute_validation_loss_tolerance': TRANSFER_ABS_TOL,
    'output': str(OUTPUT),
}))


## 课中 1：运行全部宽度与候选

默认训练60步，命令行脚本默认120步；这改变了有限预算下的验证曲线，应在报告中明确。设备设为 `auto`，自动使用可用CUDA或CPU。所有宽度、参数化和候选共享同一批次索引序列；初始化使用嵌套的原始神经元样本。

失败配置仍保留在预声明网格中。报告中的 `selection_at_grid_boundary` 提醒最优候选是否落在端点：端点最优时，应预先声明一个扩展扫描来检查，不能把有限网格最优说成全局最优。


In [ ]:
report = run_lab(config, output=OUTPUT)
display(Image(filename=str(OUTPUT / 'lab03_width.png')))
display(Image(filename=str(OUTPUT / 'feature_dynamics.png')))


In [ ]:
saved_report = json.loads((OUTPUT / 'report.json').read_text(encoding='utf-8'))
failed = [
    {key: row[key] for key in ('regime', 'width', 'base_lr', 'raw_lr', 'status')}
    for row in saved_report['summaries'] if row['status'] != 'completed'
]
display(JSON({
    'actual_device': saved_report['actual_device'],
    'config': saved_report['config'],
    'selected': saved_report['selected'],
    'failed_configurations': failed,
    'conventions': saved_report['conventions'],
}))


## 课中 2：核对初始化和诊断的归一化

`initial_normalized_probe_kernel_trace` 是固定探针集上

$$\Theta_{\mathrm{probe}}=\frac{p_m}{n_{\mathrm{probe}}}JJ^\top$$

的迹，其中 $p_m$ 是原始学习率相对基础学习率的倍率。它已经除以探针样本数，并不是未归一化的全训练集核，不能直接拿它代替训练核的最大特征值来判断稳定边界。

下面核对同一宽度、同一原始初始化下的两个量：中心化初始输出应接近数值零；两种参数化的初始归一化有效核迹应在舍入误差内一致。原始未中心化输出RMS可以不同。


In [ ]:
initial_audit = []
for width in config.widths:
    pair = {}
    for regime in ('NTP', 'MF'):
        row = next(r for r in report['summaries']
                   if r['regime'] == regime and r['width'] == width)
        pair[regime] = {
            'raw_initial_output_rms': row['raw_initial_output_rms'],
            'centered_initial_output_rms': row['centered_initial_output_rms'],
            'normalized_probe_kernel_trace': row['initial_normalized_probe_kernel_trace'],
        }
    pair['trace_difference'] = (
        pair['NTP']['normalized_probe_kernel_trace'] - pair['MF']['normalized_probe_kernel_trace']
    )
    initial_audit.append({'width': width, **pair})
display(JSON(initial_audit))
fixed_eta = config.rates[len(config.rates) // 2]
print('脚本特征变化与过程图使用同一个基础学习率：', fixed_eta)
print('诊断分母 epsilon_diag：', config.diagnostic_epsilon)


## 课中 3：用预声明容忍量检查迁移

以下只检查**当前候选网格内的验证迁移**：先在最小宽度按验证集选基础学习率，再在其他宽度查找同一个学习率的结果，并与各宽度自己的扫描最优值比较。代码不读取测试损失来选择阈值或学习率。

`within_declared_tolerance=True` 只意味着通过该有限网格、固定预算的检查。若小宽度或大宽度的最佳候选位于网格边界，仍须另外检查扫描范围；若某个配置失败，结果保留为失败而不从分母中消失。


In [ ]:
def check_transfer(item, tolerance):
    widths = tuple(item['config']['widths'])
    source_width = min(widths)
    checks = []
    for regime in ('NTP', 'MF'):
        source = item['selected'][f'{regime}:{source_width}']
        chosen_lr = source['base_lr']
        for width in widths:
            best = item['selected'][f'{regime}:{width}']
            candidate = next(
                (r for r in item['summaries']
                 if r['regime'] == regime and r['width'] == width and r['base_lr'] == chosen_lr),
                None,
            )
            usable = (
                candidate is not None and candidate['status'] == 'completed'
                and candidate['validation_loss'] is not None and best['validation_loss'] is not None
            )
            gap = candidate['validation_loss'] - best['validation_loss'] if usable else None
            checks.append({
                'regime': regime, 'source_width': source_width, 'target_width': width,
                'source_selected_base_lr': chosen_lr,
                'source_selection_at_grid_boundary': source['selection_at_grid_boundary'],
                'target_selection_at_grid_boundary': best['selection_at_grid_boundary'],
                'candidate_status': candidate['status'] if candidate is not None else 'not_available',
                'validation_gap_to_target_grid_best': gap,
                'absolute_tolerance': tolerance,
                'within_declared_tolerance': bool(usable and gap <= tolerance),
            })
    return checks

transfer_checks = check_transfer(report, TRANSFER_ABS_TOL)
display(JSON(transfer_checks))


## 课中 4：查看过程记录，区分几种变化

`training_curves.csv` 保存每个检查点的训练/验证损失、$D_W(t)$ 与 $D_C(t)$。`width_summary.csv` 还保存最终的特征相对变化和有效核相对变化。这些量不同：隐藏Gram只测隐藏特征的二阶关系；有效核还包含输出系数、激活导数与学习率倍率。


In [ ]:
import csv

with (OUTPUT / 'training_curves.csv').open(encoding='utf-8', newline='') as handle:
    records = list(csv.DictReader(handle))
example_records = [
    row for row in records
    if row['regime'] == 'MF' and int(row['width']) == min(config.widths)
    and float(row['base_lr']) == fixed_eta
]
display(JSON({'first_checkpoints': example_records[:3], 'last_checkpoints': example_records[-3:]}))
print('所有候选的最终诊断：', OUTPUT / 'width_summary.csv')


## 课后扩展：至少三个随机种子

将下面的开关改为 `True`，保持宽度、步数、batch、六个候选和迁移容忍量不变，运行种子2027、2028、2029。每个种子都有独立结果目录。

当前脚本的种子同时改变数据采样、初始化及batch流；因此聚合结果包含这三类随机性，而不是只隔离初始化。每个种子内部的不同方法仍共享同一协议。


In [ ]:
RUN_MULTIPLE_SEEDS = False
SEEDS = (2027, 2028, 2029)
multi_reports = {}
if RUN_MULTIPLE_SEEDS:
    for seed in SEEDS:
        seeded_config = replace(config, seed=seed)
        seeded_output = COURSE_ROOT / 'results' / 'notebooks' / 'lab03' / f'multi_seed_{seed}'
        multi_reports[seed] = run_lab(seeded_config, output=seeded_output)
        print('已完成种子', seed, '设备', multi_reports[seed]['actual_device'])
else:
    print('三种子扩展默认关闭；当前只完成单种子有限扫描，不据此声明参数化的普遍优劣。')


In [ ]:
from statistics import mean, stdev

if multi_reports:
    transfer_by_seed = {
        str(seed): check_transfer(item, TRANSFER_ABS_TOL)
        for seed, item in multi_reports.items()
    }
    aggregated = []
    keys = sorted(next(iter(multi_reports.values()))['selected'])
    for key in keys:
        choices = [item['selected'][key] for item in multi_reports.values()]
        values = [v['validation_loss'] for v in choices if v['validation_loss'] is not None]
        aggregated.append({
            'regime_and_width': key,
            'declared_seed_count': len(choices),
            'successful_seed_count': len(values),
            'mean_grid_selected_validation_loss': mean(values) if values else None,
            'sample_standard_deviation': stdev(values) if len(values) > 1 else None,
            'boundary_selection_count': sum(bool(v['selection_at_grid_boundary']) for v in choices),
            'selected_base_rates': [v['base_lr'] for v in choices],
        })
    display(JSON({'descriptive_summary': aggregated, 'transfer_checks_by_seed': transfer_by_seed}))
else:
    print('尚未运行多种子扩展；启用上一格后可比较趋势、边界选择和失败配置。')


## 提交问题与核验要点

1. 列出实际前向、存储变量、原始学习率、损失归一化与中心化方式。证明这些约定与正文配对。
2. 在相同基础学习率下，参数RMS移动和特征Gram漂移如何随宽度变化？这些有限宽度结果是否支持你的课前预测？
3. 按预声明容忍量报告迁移检查。若最优点在网格边界、部分配置失败或不同种子不一致，应怎样限定结论？
4. 为什么“有效核近似固定但训练损失下降”不是矛盾？为什么“特征漂移较大”不等于“泛化较好”？
5. 如果mean-field越宽越慢，先检查哪些尺度与实现约定，再考虑模型条件或有限时间效应？

**核验标准：** 保存全部候选，固定预算，不使用测试结果调节容忍量；报告至少三个种子并说明随机性来源。当前两层SGD教学约定不是任意深层架构的完整μP配方，也不能直接照搬到Adam或Muon。
